# 67 — P10.7 SPIDER: entrenamiento multitarea degenerativo por nivel discal

    **Tipo de entrenamiento fijado**

    - Clasificación supervisada multitarea por nivel discal.
    - Dos ramas 2.5D independientes: T1 y T2, sin asumir registro píxel a píxel.
    - Encoder compartido EfficientNet-B0 con transferencia ImageNet.
    - Fusión tardía de embeddings T1/T2, disponibilidad de secuencia y embedding del nivel IVD.
    - Ocho cabezas: Pfirrmann, Modic, platillo superior, platillo inferior, espondilolistesis,
      hernia, estrechamiento y abombamiento.
    - Selección exclusivamente con `dev_val`; `internal_test` permanece sellado.


> **Gobernanza obligatoria**
>
> - No reentrena estenosis central, foraminal ni subarticular: esas tareas P10.6 ya tienen checkpoints.
> - No accede al test oculto de SPIDER.
> - No usa el `internal_test` para seleccionar modelo o ajustar hiperparámetros.
> - Antes de escribir resultados audita notebooks, manifests, resultados, modelos y todos los `.pt`.
> - Si detecta un export final/frozen previo de P10.7, aborta.
> - La salida es de investigación, requiere revisión profesional y no constituye diagnóstico clínico.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip -q install timm>=1.0.0 scikit-learn>=1.3.0


In [3]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import timm
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, average_precision_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 2026

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

PFI_ROOT = Path(os.getenv("PFI_ROOT", "/content/drive/MyDrive/PFI_MVP"))
RESULTS_ROOT = Path(os.getenv(
    "PFI_P10_7_RESULTS_ROOT",
    str(PFI_ROOT / "results" / "P10_7_spider_degenerative")
))
MODELS_ROOT = Path(os.getenv(
    "PFI_P10_7_MODELS_ROOT",
    str(PFI_ROOT / "models" / "P10_7_spider_degenerative")
))
MANIFEST_PATH = RESULTS_ROOT / "disc_level_manifest_v1.csv"
SCOPE_PATH = RESULTS_ROOT / "task_scope_v1.json"
NB66_MARKER = RESULTS_ROOT / "NOTEBOOK_66_COMPLETE.json"
CHECKPOINT_DIR = MODELS_ROOT / "checkpoints"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best_candidate.pt"
TRAINING_SUMMARY = RESULTS_ROOT / "training_summary_v1.json"
COMPLETE_PATH = RESULTS_ROOT / "NOTEBOOK_67_COMPLETE.json"

for path in [MANIFEST_PATH, SCOPE_PATH, NB66_MARKER]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta requisito: {path}")

final_pt = [
    p for p in MODELS_ROOT.rglob("*.pt")
    if any(token in p.name.lower() for token in ("final", "frozen", "research_export"))
]
if final_pt:
    raise RuntimeError(f"P10.7 ya tiene export final. No se reentrena: {final_pt}")

allow_resume = os.getenv("PFI_P10_7_ALLOW_RESUME", "NO").strip().upper() == "YES"
if BEST_CHECKPOINT.exists() and not allow_resume:
    raise RuntimeError(
        f"Ya existe {BEST_CHECKPOINT}. Para reanudar de forma explícita usar "
        "PFI_P10_7_ALLOW_RESUME=YES; no se sobrescribe silenciosamente."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [4]:
@dataclass
class TrainConfig:
    image_size: int = 224
    backbone: str = "efficientnet_b0"
    pretrained: bool = True
    batch_size: int = 16
    num_workers: int = 2
    epochs: int = 25
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    patience: int = 6
    dropout: float = 0.25
    gradient_clip: float = 1.0

CFG = TrainConfig(
    batch_size=int(os.getenv("PFI_P10_7_BATCH_SIZE", "16")),
    epochs=int(os.getenv("PFI_P10_7_EPOCHS", "25")),
    num_workers=int(os.getenv("PFI_P10_7_NUM_WORKERS", "2")),
    pretrained=os.getenv("PFI_P10_7_IMAGENET_PRETRAINED", "1") == "1",
)
CFG


TrainConfig(image_size=224, backbone='efficientnet_b0', pretrained=True, batch_size=16, num_workers=2, epochs=25, learning_rate=0.0002, weight_decay=0.0001, patience=6, dropout=0.25, gradient_clip=1.0)

In [5]:
TASK_ORDER = [
    "pfirrmann_grade",
    "modic_change",
    "upper_endplate_change",
    "lower_endplate_change",
    "spondylolisthesis",
    "disc_herniation",
    "disc_narrowing",
    "disc_bulging",
]
CATEGORICAL_TASKS = {
    "pfirrmann_grade": 5,
    "modic_change": 4,
}
BINARY_TASKS = [
    "upper_endplate_change",
    "lower_endplate_change",
    "spondylolisthesis",
    "disc_herniation",
    "disc_narrowing",
    "disc_bulging",
]

manifest = pd.read_csv(MANIFEST_PATH, dtype={"Patient": str})
forbidden = manifest[manifest["split"].eq("internal_test")]
dev = manifest[manifest["split"].isin(["dev_train", "dev_val"])].copy()

if dev.empty:
    raise ValueError("No hay muestras dev_train/dev_val.")
if set(dev["Patient"]) & set(forbidden["Patient"]):
    raise ValueError("Leakage de pacientes entre desarrollo e internal_test.")

print(dev["split"].value_counts())
print("internal_test sellado:", len(forbidden), "muestras")


split
dev_train    992
dev_val      248
Name: count, dtype: int64
internal_test sellado: 278 muestras


In [12]:
class DiscCropDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, augment: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.frame)

    def _augment(self, tensor: torch.Tensor) -> torch.Tensor:
        if not self.augment:
            return tensor
        # No se usa flip geométrico para preservar orientación.
        if torch.rand(()) < 0.5:
            scale = 0.90 + 0.20 * torch.rand(())
            bias = -0.05 + 0.10 * torch.rand(())
            tensor = torch.clamp(tensor * scale + bias, 0.0, 1.0)
        if torch.rand(()) < 0.3:
            tensor = torch.clamp(tensor + torch.randn_like(tensor) * 0.015, 0.0, 1.0)
        return tensor

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        row = self.frame.iloc[index]
        payload = np.load(row["crop_path"])
        t1 = self._augment(torch.from_numpy(payload["t1"]).float())
        t2 = self._augment(torch.from_numpy(payload["t2"]).float())
        availability = torch.from_numpy(payload["availability"]).float()
        ivd_index = torch.tensor(int(row["ivd_label"]) - 1, dtype=torch.long)

        labels = torch.tensor([
            float(row[task]) if pd.notna(row[task]) else -1.0
            for task in TASK_ORDER
        ], dtype=torch.float32)
        label_mask = torch.tensor([
            1.0 if pd.notna(row[task]) else 0.0
            for task in TASK_ORDER
        ], dtype=torch.float32)

        return {
            "t1": t1,
            "t2": t2,
            "availability": availability,
            "ivd_index": ivd_index,
            "labels": labels,
            "label_mask": label_mask,
        }

train_ds = DiscCropDataset(dev[dev["split"].eq("dev_train")], augment=True)
val_ds = DiscCropDataset(dev[dev["split"].eq("dev_val")], augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
print("train:", len(train_ds), "val:", len(val_ds))


train: 992 val: 248


In [7]:
class MultiTaskDiscClassifier(nn.Module):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        try:
            self.encoder = timm.create_model(
                cfg.backbone,
                pretrained=cfg.pretrained,
                num_classes=0,
                global_pool="avg",
                in_chans=3,
            )
        except Exception as exc:
            raise RuntimeError(
                "No se pudo inicializar el backbone solicitado. "
                "No se cambia silenciosamente a entrenamiento desde cero."
            ) from exc

        feature_dim = int(self.encoder.num_features)
        self.ivd_embedding = nn.Embedding(25, 16)
        fused_dim = feature_dim * 2 + 2 + 16
        self.trunk = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
        )
        self.heads = nn.ModuleDict({
            "pfirrmann_grade": nn.Linear(256, 5),
            "modic_change": nn.Linear(256, 4),
            **{task: nn.Linear(256, 1) for task in BINARY_TASKS},
        })

    def forward(
        self,
        t1: torch.Tensor,
        t2: torch.Tensor,
        availability: torch.Tensor,
        ivd_index: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        t1_features = self.encoder(t1)
        t2_features = self.encoder(t2)
        t1_features = t1_features * availability[:, 0:1]
        t2_features = t2_features * availability[:, 1:2]
        ivd_features = self.ivd_embedding(ivd_index.clamp(0, 24))
        fused = torch.cat([t1_features, t2_features, availability, ivd_features], dim=1)
        shared = self.trunk(fused)
        return {name: head(shared) for name, head in self.heads.items()}

model = MultiTaskDiscClassifier(CFG).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

parameters: 5464603


In [20]:
train_frame = dev[dev["split"].eq("dev_train")]

CATEGORICAL_WEIGHT_MIN = float(os.getenv("PFI_P10_7_CATEGORICAL_WEIGHT_MIN", "0.35"))
CATEGORICAL_WEIGHT_MAX = float(os.getenv("PFI_P10_7_CATEGORICAL_WEIGHT_MAX", "3.0"))
BINARY_POS_WEIGHT_MIN = float(os.getenv("PFI_P10_7_BINARY_POS_WEIGHT_MIN", "0.35"))
BINARY_POS_WEIGHT_MAX = float(os.getenv("PFI_P10_7_BINARY_POS_WEIGHT_MAX", "5.0"))
CLASS_WEIGHT_SMOOTHING = float(os.getenv("PFI_P10_7_CLASS_WEIGHT_SMOOTHING", "1.0"))

def smoothed_class_weights(values: pd.Series, n_classes: int) -> np.ndarray:
    counts = values.value_counts().reindex(range(n_classes), fill_value=0).astype(float).values
    smoothed_counts = counts + CLASS_WEIGHT_SMOOTHING
    raw = smoothed_counts.sum() / (smoothed_counts * n_classes)
    weights = np.sqrt(raw)
    weights = weights / max(float(weights.mean()), 1e-6)
    return np.clip(weights, CATEGORICAL_WEIGHT_MIN, CATEGORICAL_WEIGHT_MAX).astype(np.float32)

def smoothed_positive_weight(values: pd.Series) -> float:
    positives = float((values == 1).sum()) + CLASS_WEIGHT_SMOOTHING
    negatives = float((values == 0).sum()) + CLASS_WEIGHT_SMOOTHING
    raw = negatives / max(positives, 1e-6)
    return float(np.clip(math.sqrt(raw), BINARY_POS_WEIGHT_MIN, BINARY_POS_WEIGHT_MAX))

weight_audit: dict[str, Any] = {
    "schemaVersion": "pfi.p10-7-training-weights.v2",
    "categoricalWeightRange": [CATEGORICAL_WEIGHT_MIN, CATEGORICAL_WEIGHT_MAX],
    "binaryPositiveWeightRange": [BINARY_POS_WEIGHT_MIN, BINARY_POS_WEIGHT_MAX],
    "classWeightSmoothing": CLASS_WEIGHT_SMOOTHING,
    "tasks": {},
}

categorical_weights: dict[str, torch.Tensor] = {}
for task, n_classes in CATEGORICAL_TASKS.items():
    values = train_frame[task].dropna().astype(int)
    counts = values.value_counts().reindex(range(n_classes), fill_value=0).astype(int)
    weights = smoothed_class_weights(values, n_classes)
    categorical_weights[task] = torch.tensor(weights, dtype=torch.float32, device=device)
    weight_audit["tasks"][task] = {
        "type": "categorical",
        "counts": {str(k): int(v) for k, v in counts.items()},
        "weights": {str(idx): float(value) for idx, value in enumerate(weights)},
    }

binary_pos_weight: dict[str, torch.Tensor] = {}
for task in BINARY_TASKS:
    values = train_frame[task].dropna().astype(int)
    positives = int((values == 1).sum())
    negatives = int((values == 0).sum())
    weight = smoothed_positive_weight(values)
    binary_pos_weight[task] = torch.tensor([weight], dtype=torch.float32, device=device)
    weight_audit["tasks"][task] = {
        "type": "binary",
        "positives": positives,
        "negatives": negatives,
        "posWeight": weight,
    }

print(json.dumps(weight_audit, indent=2, ensure_ascii=False))

task_to_index = {task: idx for idx, task in enumerate(TASK_ORDER)}

def multitask_loss(
    outputs: dict[str, torch.Tensor],
    labels: torch.Tensor,
    label_mask: torch.Tensor,
) -> tuple[torch.Tensor, dict[str, float]]:
    losses = []
    details = {}

    for task, n_classes in CATEGORICAL_TASKS.items():
        idx = task_to_index[task]
        active = label_mask[:, idx].bool()
        if active.any():
            loss = nn.functional.cross_entropy(
                outputs[task][active],
                labels[active, idx].long(),
                weight=categorical_weights[task],
            )
            losses.append(loss)
            details[task] = float(loss.detach().cpu())

    for task in BINARY_TASKS:
        idx = task_to_index[task]
        active = label_mask[:, idx].bool()
        if active.any():
            loss = nn.functional.binary_cross_entropy_with_logits(
                outputs[task][active, 0],
                labels[active, idx],
                pos_weight=binary_pos_weight[task],
            )
            losses.append(loss)
            details[task] = float(loss.detach().cpu())

    if not losses:
        raise RuntimeError("Batch sin etiquetas válidas.")
    return torch.stack(losses).mean(), details


{
  "schemaVersion": "pfi.p10-7-training-weights.v2",
  "categoricalWeightRange": [
    0.35,
    3.0
  ],
  "binaryPositiveWeightRange": [
    0.35,
    5.0
  ],
  "classWeightSmoothing": 1.0,
  "tasks": {
    "pfirrmann_grade": {
      "type": "categorical",
      "counts": {
        "0": 188,
        "1": 246,
        "2": 274,
        "3": 168,
        "4": 116
      },
      "weights": {
        "0": 0.9935891032218933,
        "1": 0.8691390156745911,
        "2": 0.8237043619155884,
        "3": 1.0507378578186035,
        "4": 1.2628296613693237
      }
    },
    "modic_change": {
      "type": "categorical",
      "counts": {
        "0": 694,
        "1": 4,
        "2": 288,
        "3": 6
      },
      "weights": {
        "0": 0.3499999940395355,
        "1": 1.9403284788131714,
        "2": 0.3499999940395355,
        "3": 1.6398769617080688
      }
    },
    "upper_endplate_change": {
      "type": "binary",
      "positives": 391,
      "negatives": 601,
      "posWe

In [21]:
@torch.no_grad()
def evaluate(loader: DataLoader) -> dict[str, Any]:
    model.eval()
    collected = {
        task: {"y": [], "score": [], "pred": []}
        for task in TASK_ORDER
    }
    losses = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            batch["t1"],
            batch["t2"],
            batch["availability"],
            batch["ivd_index"],
        )
        loss, _ = multitask_loss(outputs, batch["labels"], batch["label_mask"])
        losses.append(float(loss.cpu()))

        for task in TASK_ORDER:
            idx = task_to_index[task]
            active = batch["label_mask"][:, idx].bool()
            if not active.any():
                continue
            y = batch["labels"][active, idx].detach().cpu().numpy()
            if task in CATEGORICAL_TASKS:
                prob = torch.softmax(outputs[task][active], dim=1).detach().cpu().numpy()
                pred = prob.argmax(axis=1)
                collected[task]["score"].extend(prob.tolist())
                collected[task]["pred"].extend(pred.tolist())
            else:
                prob = torch.sigmoid(outputs[task][active, 0]).detach().cpu().numpy()
                pred = (prob >= 0.5).astype(int)
                collected[task]["score"].extend(prob.tolist())
                collected[task]["pred"].extend(pred.tolist())
            collected[task]["y"].extend(y.tolist())

    metrics: dict[str, Any] = {"loss": float(np.mean(losses))}
    selection_components = []

    for task, values in collected.items():
        y = np.asarray(values["y"])
        pred = np.asarray(values["pred"])
        if y.size == 0:
            metrics[task] = {"status": "no_labels"}
            continue

        if task in CATEGORICAL_TASKS:
            macro_f1 = float(f1_score(y, pred, average="macro", zero_division=0))
            balanced = float(balanced_accuracy_score(y, pred))
            metrics[task] = {
                "macroF1": macro_f1,
                "balancedAccuracy": balanced,
                "support": int(y.size),
            }
            selection_components.append((macro_f1 + balanced) / 2.0)
        else:
            score = np.asarray(values["score"], dtype=float)
            macro_f1 = float(f1_score(y, pred, average="binary", zero_division=0))
            result = {"f1": macro_f1, "support": int(y.size)}
            if len(np.unique(y)) == 2:
                result["rocAuc"] = float(roc_auc_score(y, score))
                result["averagePrecision"] = float(average_precision_score(y, score))
                selection_components.append(
                    (macro_f1 + result["rocAuc"] + result["averagePrecision"]) / 3.0
                )
            else:
                result["rocAuc"] = None
                result["averagePrecision"] = None
                selection_components.append(macro_f1)
            metrics[task] = result

    metrics["selectionScore"] = float(np.mean(selection_components))
    return metrics


In [22]:
import time

started = time.time()
batch = next(iter(train_loader))

print("Primer batch cargado en:", round(time.time() - started, 2), "s")
print("t1:", batch["t1"].shape)
print("t2:", batch["t2"].shape)
print("availability:", batch["availability"].shape)
print("labels:", batch["labels"].shape)

Primer batch cargado en: 0.2 s
t1: torch.Size([16, 3, 224, 224])
t2: torch.Size([16, 3, 224, 224])
availability: torch.Size([16, 2])
labels: torch.Size([16, 8])


In [23]:
from tqdm.auto import tqdm

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

start_epoch = 1
best_score = -math.inf
best_metrics = None
stale_epochs = 0
history = []

if BEST_CHECKPOINT.exists() and allow_resume:
    checkpoint = torch.load(
        BEST_CHECKPOINT,
        map_location=device,
        weights_only=False,
    )
    model.load_state_dict(
        checkpoint["modelStateDict"],
        strict=True,
    )
    optimizer.load_state_dict(
        checkpoint["optimizerStateDict"]
    )

    start_epoch = int(checkpoint["epoch"]) + 1
    best_score = float(checkpoint["selectionScore"])
    best_metrics = checkpoint["validationMetrics"]

    print("Reanudando desde epoch", checkpoint["epoch"])

print("==============================================")
print("INICIO DEL ENTRENAMIENTO")
print("==============================================")
print("device:", device)
print("epochs máximas:", CFG.epochs)
print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("batch size:", CFG.batch_size)
print("num_workers:", CFG.num_workers)
print()

for epoch in range(start_epoch, CFG.epochs + 1):
    model.train()
    train_losses = []
    started = time.time()

    print(f"\n{'=' * 55}")
    print(f"EPOCH {epoch}/{CFG.epochs}")
    print(f"{'=' * 55}")

    train_progress = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"Epoch {epoch:02d} · train",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )

    for batch_index, batch in enumerate(
        train_progress,
        start=1,
    ):
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            batch["t1"],
            batch["t2"],
            batch["availability"],
            batch["ivd_index"],
        )

        loss, _ = multitask_loss(
            outputs,
            batch["labels"],
            batch["label_mask"],
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Loss no finita en epoch={epoch}, "
                f"batch={batch_index}: {loss.item()}"
            )

        loss.backward()

        gradient_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG.gradient_clip,
        )

        optimizer.step()

        loss_value = float(loss.detach().cpu())
        train_losses.append(loss_value)

        recent_loss = float(
            np.mean(train_losses[-10:])
        )

        gpu_memory_gb = (
            torch.cuda.memory_allocated() / 1024**3
            if torch.cuda.is_available()
            else 0.0
        )

        train_progress.set_postfix(
            loss=f"{recent_loss:.4f}",
            grad=f"{float(gradient_norm):.3f}",
            gpu=f"{gpu_memory_gb:.1f}GB",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
        )

    print("\nEntrenamiento de la época terminado.")
    print("Iniciando validación...")

    validation_progress = tqdm(
        val_loader,
        total=len(val_loader),
        desc=f"Epoch {epoch:02d} · validation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )

    val_metrics = evaluate(validation_progress)

    score = float(val_metrics["selectionScore"])
    scheduler.step(score)

    elapsed_seconds = time.time() - started
    mean_train_loss = float(np.mean(train_losses))

    epoch_record = {
        "epoch": epoch,
        "trainLoss": mean_train_loss,
        "validation": val_metrics,
        "learningRate": float(
            optimizer.param_groups[0]["lr"]
        ),
        "seconds": elapsed_seconds,
    }

    history.append(epoch_record)

    print("\n----------------------------------------------")
    print(f"Epoch completada:     {epoch}/{CFG.epochs}")
    print(f"Train loss:           {mean_train_loss:.6f}")
    print(f"Selection score:      {score:.6f}")
    print(f"Mejor score anterior: {best_score:.6f}")
    print(f"Duración:              {elapsed_seconds / 60:.2f} min")
    print(
        "Learning rate:        "
        f"{optimizer.param_groups[0]['lr']:.2e}"
    )
    print("----------------------------------------------")

    # Conserva el JSON completo original por trazabilidad.
    print(json.dumps(epoch_record, indent=2))

    if score > best_score:
        best_score = score
        best_metrics = copy.deepcopy(val_metrics)
        stale_epochs = 0

        torch.save(
            {
                "schemaVersion":
                    "pfi.p10-7-training-checkpoint.v1",
                "epoch": epoch,
                "modelStateDict": model.state_dict(),
                "optimizerStateDict":
                    optimizer.state_dict(),
                "selectionScore": best_score,
                "validationMetrics": best_metrics,
                "trainConfig": asdict(CFG),
                "taskOrder": TASK_ORDER,
                "categoricalTasks": CATEGORICAL_TASKS,
                "binaryTasks": BINARY_TASKS,
                "seed": SEED,
                "humanReviewRequired": True,
                "notClinicalDiagnosis": True,
                "internalTestAccessed": False,
                "officialHiddenTestAccessed": False,
            },
            BEST_CHECKPOINT,
        )

        print(
            "\nNUEVO MEJOR CHECKPOINT GUARDADO:",
            BEST_CHECKPOINT,
        )
        print("Mejor selectionScore:", best_score)

    else:
        stale_epochs += 1

        print(
            f"\nSin mejora: {stale_epochs}/{CFG.patience}"
        )

        if stale_epochs >= CFG.patience:
            print("Early stopping.")
            break

print("\n==============================================")
print("ENTRENAMIENTO FINALIZADO")
print("==============================================")
print("Mejor selectionScore:", best_score)
print("Checkpoint:", BEST_CHECKPOINT)

NameError: name 'model' is not defined

In [ ]:
import gc
import torch

CFG.num_workers = 0

try:
    del train_loader, val_loader
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("num_workers:", CFG.num_workers)

In [ ]:
if not BEST_CHECKPOINT.exists():
    raise RuntimeError("No se generó best_candidate.pt")

digest = hashlib.sha256()
with BEST_CHECKPOINT.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(chunk)

summary = {
    "schemaVersion": "pfi.p10-7-training-summary.v1",
    "status": "CANDIDATE_SELECTED_ON_DEV_VAL",
    "bestCandidatePath": str(BEST_CHECKPOINT),
    "bestCandidateSha256": digest.hexdigest(),
    "bestSelectionScore": best_score,
    "bestValidationMetrics": best_metrics,
    "history": history,
    "trainConfig": asdict(CFG),
    "internalTestAccessed": False,
    "officialHiddenTestAccessed": False,
    "readyForInternalTest": True,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}
TRAINING_SUMMARY.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    encoding="utf-8",
)
COMPLETE_PATH.write_text(
    json.dumps({
        "status": "NOTEBOOK_67_COMPLETE",
        "bestCandidateSha256": digest.hexdigest(),
        "internalTestAccessed": False,
    }, indent=2) + "\n",
    encoding="utf-8",
)

print("NOTEBOOK_67_COMPLETE")
print("best candidate:", BEST_CHECKPOINT)
print("sha256:", digest.hexdigest())
